# PopOut — Adversarial Search Strategies
**Artificial Intelligence 2025/2026** | DCC – FCUP

PopOut is a variant of Connect-4 where players can also remove a disc
from the bottom of a column (pop move), shifting all pieces above it
down by one row. The first player to connect four discs wins.

**Group members**
- Daniel Viloria Prieto
- (member 2)
- (member 3)

---

## 1. Imports

In [1]:
from src.game.board import Board, PLAYER1, PLAYER2

from src.game.player import (
    HumanPlayer, RandomPlayer,
    MCTSPlayer, MCTSPlayerV2, MCTSPlayerV3,
    MCTSPlayerV4, MCTSPlayerV5, MCTSPlayerV6, DecisionTreePlayer
)

from src.game.game import Game
from src.decision_tree.id3 import ID3DecisionTree
from src.decision_tree.tree import DecisionTreeNode
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # non-interactive backend for headless execution
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

## 2. Game Overview

The board is a 6×7 grid. Each turn a player can:
- **Drop** — place a disc into a column from the top
- **Pop** — remove their own disc from the bottom of a column

Three additional rules handle edge cases:
- If a pop creates four-in-a-row for both players, the player who popped wins
- If the board is full and a player can pop, they may declare a draw instead
- If the same state repeats three times, either player may claim a draw

## Human vs Human

In [ ]:
# Uncomment to play Human vs Human (interactive session only)
# player1 = HumanPlayer(PLAYER1)
# player2 = HumanPlayer(PLAYER2)

# game = Game(player1, player2)

# game.play()

## 4. Human vs Computer (MCTS)

The MCTS algorithm builds a search tree by iterating four phases:
1. **Selection** — descend using UCT to find a promising node
2. **Expansion** — expand one untried move
3. **Simulation** — play out a random game from that node
4. **Backpropagation** — update win/visit counts up the tree

Six variants are available, differing in simulation strategy,
final move selection (max visits vs max wins), and tree reuse.

| Version | Simulation | Final selection | Tree reuse |
|---------|-----------|----------------|------------|
| V1 | Random | Max visits | No |
| V2 | Random | Max visits | Yes |
| V3 | Smart | Max visits | No |
| V4 | Random | Max wins | No |
| V5 | Random | Max wins | Yes |
| V6 | Smart | Max wins | No |

In [ ]:
player1 = HumanPlayer(PLAYER1)
player2 = MCTSPlayerV3(PLAYER2, iterations=2000)

game = Game(player1, player2)

game.play()

## 5. Computer vs Computer (MCTS versions)

In [ ]:
player1 = MCTSPlayer(PLAYER1, iterations=5000)
player2 = MCTSPlayerV6(PLAYER2, iterations=5000)

game = Game(player1, player2)

game.play()

## 7. Decision Tree (ID3)
The ID3 procedure builds a decision tree by recursively selecting the attribute that maximises information gain (entropy reduction).

For continuous attributes we find the optimal split threshold automatically.

### 7.1 Dataset 1 - Iris (warm-up)


In [ ]:
# Load the iris dataset
from src.game.dataset_generator import load_iris
X_iris, y_iris, iris_features = load_iris('data/iris.csv')

print(f'Iris dataset: {len(X_iris)} samples, {len(iris_features)} features')
print('Features:', iris_features)
print('Classes:', sorted(set(y_iris)))

df = pd.DataFrame(X_iris, columns=iris_features)
df['class'] = y_iris
df.describe()

In [ ]:
# Train / test split (80 / 20)
import random
random.seed(42)
indices = list(range(len(X_iris)))
random.shuffle(indices)
split = int(0.8 * len(indices))

X_train = [X_iris[i] for i in indices[:split]]
y_train = [y_iris[i] for i in indices[:split]]
X_test  = [X_iris[i] for i in indices[split:]]
y_test  = [y_iris[i] for i in indices[split:]]

print(f'Train: {len(X_train)} samples  |  Test: {len(X_test)} samples')

In [ ]:
# Build the ID3 decision tree (all four features are continuous)
iris_tree = ID3DecisionTree(
    continuous_features=list(range(4)),
    feature_names=iris_features,
)
iris_tree.fit(X_train, y_train)

print('=== Iris Decision Tree ===')
iris_tree.display()

train_eval = iris_tree.evaluate(X_train, y_train)
test_eval  = iris_tree.evaluate(X_test,  y_test)
print(f'\nTrain accuracy: {train_eval["accuracy"]:.2%}  ({train_eval["correct"]}/{train_eval["total"]})')
print(f'Test  accuracy: {test_eval["accuracy"]:.2%}  ({test_eval["correct"]}/{test_eval["total"]})')

In [ ]:
# Effect of max_depth on Iris accuracy
depths = [1, 2, 3, 4, 5, None]
train_accs, test_accs = [], []

for d in depths:
    t = ID3DecisionTree(continuous_features=list(range(4)), max_depth=d)
    t.fit(X_train, y_train)
    train_accs.append(t.evaluate(X_tfallback=RandomPlayer()rain, y_train)['accuracy'])
    test_accs.append(t.evaluate(X_test,   y_test)['accuracy'])

depth_labels = [str(d) if d is not None else 'None' for d in depths]
x = range(len(depths))

plt.figure(figsize=(8, 4))
plt.plot(x, train_accs, marker='o', label='Train')
plt.plot(x, test_accs,  marker='s', label='Test')
plt.xticks(x, depth_labels)
plt.xlabel('max_depth')
plt.ylabel('Accuracy')
plt.title('Iris — Effect of max_depth on ID3 accuracy')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('iris_depth.png', dpi=80)
plt.show()
plt.close()
print('Plot saved to iris_depth.png')

### 7.2 Dataset 2 - PopOut Game
We generate a labelled dataset using MCTS self-play. Each sample is a board state (42 cells + 1 current player = 43 features). The label is the MCTS-recommended move as movetype_col (e.g. d3).

In [ ]:
from src.game.dataset_generator import generate_dataset

generate_dataset(100, "data/d1.csv", 100)
df = pd.read_csv("data/dataset.csv")

print(f"Total samples: {len(df)}")
print(f"Drop moves: {(df['move_type'] == 'drop').sum()}")
print(f"Pop moves:  {(df['move_type'] == 'pop').sum()}")

df.head()

In [3]:
# Load the popout dataset
from src.game.dataset_generator import load_popout
X_popout, y_popout, popout_features = load_popout('data/popout.csv')

print(f'PopOut dataset: {len(X_popout)} samples, {len(popout_features)} features')
print('Features:', popout_features)
print('Classes:', sorted(set(y_popout)))

df = pd.DataFrame(X_popout, columns=popout_features)
df['class'] = y_popout
df.describe()

PopOut dataset: 1800 samples, 43 features
Features: ['cell_0', 'cell_1', 'cell_2', 'cell_3', 'cell_4', 'cell_5', 'cell_6', 'cell_7', 'cell_8', 'cell_9', 'cell_10', 'cell_11', 'cell_12', 'cell_13', 'cell_14', 'cell_15', 'cell_16', 'cell_17', 'cell_18', 'cell_19', 'cell_20', 'cell_21', 'cell_22', 'cell_23', 'cell_24', 'cell_25', 'cell_26', 'cell_27', 'cell_28', 'cell_29', 'cell_30', 'cell_31', 'cell_32', 'cell_33', 'cell_34', 'cell_35', 'cell_36', 'cell_37', 'cell_38', 'cell_39', 'cell_40', 'cell_41', 'current_player']
Classes: ['drop_0', 'drop_1', 'drop_2', 'drop_3', 'drop_4', 'drop_5', 'drop_6', 'pop_0', 'pop_1', 'pop_2', 'pop_3', 'pop_4', 'pop_5', 'pop_6']


,cell_0,cell_1,cell_2,cell_3,cell_4,cell_5,cell_6,cell_7,cell_8,cell_9,...,cell_33,cell_34,cell_35,cell_36,cell_37,cell_38,cell_39,cell_40,cell_41,current_player
count,1800.000000,1800.000000,1800.000000,1800.00000,1800.000000,1800.000000,1800.000000,1800.000000,1800.000000,1800.000000,...,1800.000000,1800.000000,1800.000000,1800.000000,1800.000000,1800.000000,1800.000000,1800.000000,1800.000000,1800.000000
mean,0.004444,0.012778,0.031667,0.11500,0.036667,0.034444,0.006667,0.008333,0.038889,0.126111,...,0.473333,0.273333,0.409444,0.537778,1.002222,1.071667,1.062222,0.657222,0.502778,1.481667
std,0.081551,0.134834,0.241814,0.41323,0.246803,0.260269,0.115309,0.090931,0.268069,0.475277,...,0.771476,0.657416,0.775973,0.771011,0.791840,0.478414,0.759242,0.786032,0.842185,0.499803
min,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000
25%,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,1.000000
50%,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000,0.000000,0.000000,1.000000
75%,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,1.000000,0.000000,0.000000,1.000000,2.000000,1.000000,2.000000,1.000000,1.000000,2.000000
max,2.000000,2.000000,2.000000,2.00000,2.000000,2.000000,2.000000,1.000000,2.000000,2.000000,...,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000


In [4]:
# Train / test split for PopOut dataset
import random
random.seed(0)
idx = list(range(len(X_popout)))
random.shuffle(idx)
sp = int(0.8 * len(idx))

Xp_train = [X_popout[i] for i in idx[:sp]]
yp_train = [y_popout[i] for i in idx[:sp]]
Xp_test  = [X_popout[i] for i in idx[sp:]]
yp_test  = [y_popout[i] for i in idx[sp:]]

print(f'Train: {len(Xp_train)} | Test: {len(Xp_test)}')

Train: 1440 | Test: 360


In [ ]:
# Train the ID3 decision tree on the PopOut dataset
# Board cells are categorical (values: 0, 1, 2) → no continuous_features needed
popout_tree = ID3DecisionTree(
    continuous_features=[],
    feature_names=popout_features,
    max_depth=10,
)
popout_tree.fit(Xp_train, yp_train)

pte = popout_tree.evaluate(Xp_train, yp_train)
ppe = popout_tree.evaluate(Xp_test,  yp_test)

print(f'Train accuracy: {pte["accuracy"]:.2%}')
print(f'Test  accuracy: {ppe["accuracy"]:.2%}')


Train accuracy: 89.72%
Test  accuracy: 46.39%
[cell_39]
  0:
    [cell_17]
      0:
        [cell_37]
          0:
            [cell_31]
              0:
                [cell_38]
                  0:
                    [cell_36]
                      0:
                        [cell_35]
                          0:
                            → drop_3
                          1:
                            → drop_3
                      1:
                        [cell_29]
                          0:
                            → drop_1
                          2:
                            → drop_3
                  1:
                    → drop_3
                  2:
                    → drop_3
              2:
                [cell_24]
                  0:
                    → drop_3
                  1:
                    [current_player]
                      1:
                        → drop_3
                      2:
                        → drop_3
          1:
       

In [ ]:
def run_tournament(player1, player2, n_games: int = 5, verbose: bool = False):
    """Run n_games between player1 and player2 and return win counts."""
    results = {"player1": 0, "player2": 0, "draw": 0}
    for _ in range(n_games):
        game = Game(player1, player2)
        result = game.play()
        if result == PLAYER1:
            results["player1"] += 1
        elif result == PLAYER2:
            results["player2"] += 1
        else:
            results["draw"] += 1
    return results


dt_player   = DecisionTreePlayer(PLAYER1, popout_tree)
mcts_player = MCTSPlayer(PLAYER2, 500)

print('Decision Tree player vs. MCTS player (50 iters) — 5 games')
results = run_tournament(dt_player, mcts_player, n_games=100)
print(f'  Decision Tree wins: {results["player1"]}')
print(f'  MCTS wins:          {results["player2"]}')
print(f'  Draws:              {results["draw"]}')

Decision Tree player vs. MCTS player (50 iters) — 5 games
=== PopOut Game Start ===
-------
-------
-------
-------
-------
-------
-------
-------
-------
-------
-------
---X---

 MCTSPlayer-O plays: ('drop', 3)
-------
-------
-------
-------
---O---
---X---
-------
-------
-------
---X---
---O---
---X---

 MCTSPlayer-O plays: ('drop', 0)
-------
-------
-------
---X---
---O---
O--X---
-------
-------
---X---
---X---
---O---
O--X---

 MCTSPlayer-O plays: ('drop', 2)
-------
-------
---X---
---X---
---O---
O-OX---
-------
---X---
---X---
---X---
---O---
O-OX---

 MCTSPlayer-O plays: ('drop', 6)
-------
---X---
---X---
---X---
---O---
O-OX--O
-------
---X---
---X---
---X---
---O---
O-OXX-O

 MCTSPlayer-O plays: ('drop', 1)
-------
---X---
---X---
---X---
---O---
OOOXX-O
-------
---X---
---X---
---X---
---O---
OOOXXXO

 MCTSPlayer-O plays: ('pop', 0)
-------
---X---
---X---
---X---
---O---
-OOXXXO
-------
---X---
---X---
---X---
--XO---
-OOXXXO

 MCTSPlayer-O plays: ('drop', 2)
-------